# PP-OCRv4 Mobile Rec 訓練（Colab GPU）

本筆記本用於在 Google Colab 上訓練 PP-OCRv4 mobile **文字辨識 (rec)** 模型。

**GitHub Repo:** `https://github.com/install88/trainingOcr`

**前置條件：** Det 模型已訓練完成。

**Rec 標注格式：**
```
crop_001.jpg\t2025/03/15
crop_002.jpg\tEXP:2026.01
```

**流程：**
1. Clone repo + 安裝環境
2. 上傳 rec 資料集
3. 下載預訓練模型
4. 訓練 rec 模型
5. 匯出 + 轉 ONNX
6. 下載模型

## 0. 確認 GPU

In [ ]:
!nvidia-smi

## 1. Clone Repo + 安裝環境

In [ ]:
import os
os.chdir("/content")

# Clone 專案 repo
if not os.path.exists("trainingOcr"):
    !git clone https://github.com/install88/trainingOcr.git

# Clone PaddleOCR 訓練框架
if not os.path.exists("PaddleOCR"):
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git

# 安裝 PaddlePaddle GPU 版
!pip install paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install paddleocr paddle2onnx onnxruntime shapely pyclipper imgaug lmdb

In [ ]:
import paddle
print(f"PaddlePaddle: {paddle.__version__}")
print(f"GPU available: {paddle.is_compiled_with_cuda()}")

In [ ]:
import os
os.chdir("/content")

if not os.path.exists("PaddleOCR"):
    !git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git

os.chdir("/content/PaddleOCR")
!pwd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATASET_PATH = "/content/drive/MyDrive/ocr_project/rec_dataset.zip"

!mkdir -p /content/dataset/rec
!unzip -o "{DRIVE_DATASET_PATH}" -d /content/dataset/rec/

In [ ]:
# 檢查資料集
!echo "=== 訓練集圖片數量 ==="
!ls /content/dataset/rec/train/ | wc -l
!echo "=== 驗證集圖片數量 ==="
!ls /content/dataset/rec/val/ | wc -l
!echo ""
!echo "=== train_label.txt 前 5 行 ==="
!head -5 /content/dataset/rec/train_label.txt

## 4. 下載預訓練模型

In [ ]:
!mkdir -p /content/pretrained_models

# 下載 PP-OCRv4 rec 訓練模型
!wget -nc -P /content/pretrained_models/ \
    https://paddleocr.bj.bcebos.com/PP-OCRv4/chinese/ch_PP-OCRv4_rec_train.tar

!cd /content/pretrained_models && tar xf ch_PP-OCRv4_rec_train.tar
!ls -la /content/pretrained_models/ch_PP-OCRv4_rec_train/

## 5. 建立訓練設定檔

In [ ]:
# 字元字典路徑
DICT_PATH = "/content/PaddleOCR/ppocr/utils/ppocr_keys_v1.txt"
print(f"字典路徑：{DICT_PATH}")
!wc -l {DICT_PATH}

## 5. 使用 repo 的訓練設定檔

In [ ]:
REPO_DIR = "/content/trainingOcr"
CONFIG_PATH = f"{REPO_DIR}/configs/rec/PP-OCRv4_mobile_rec_finetune.yml"
DICT_PATH = "/content/PaddleOCR/ppocr/utils/ppocr_keys_v1.txt"

!cat {CONFIG_PATH} | head -20
print(f"\n設定檔：{CONFIG_PATH}")
print(f"字典：{DICT_PATH}")
!wc -l {DICT_PATH}

In [ ]:
# [可選] 從 checkpoint 恢復訓練
# !python tools/train.py \
#     -c /content/rec_finetune.yml \
#     -o Global.checkpoints=/content/output/rec/latest

## 7. 評估模型

In [ ]:
os.chdir("/content/PaddleOCR")

!python tools/train.py \
    -c {CONFIG_PATH} \
    -o Global.use_gpu=true \
       Global.pretrained_model=/content/pretrained_models/ch_PP-OCRv4_rec_train/best_accuracy \
       Global.save_model_dir=/content/output/rec/ \
       Global.save_inference_dir=/content/output/rec/inference/ \
       Global.character_dict_path={DICT_PATH} \
       Train.dataset.data_dir=/content/dataset/rec/train/ \
       Train.dataset.label_file_list=[/content/dataset/rec/train_label.txt] \
       Eval.dataset.data_dir=/content/dataset/rec/ \
       Eval.dataset.label_file_list=[/content/dataset/rec/val_label.txt]

## 8. 匯出推論模型 + 轉 ONNX

In [ ]:
# 匯出 Paddle 推論模型
!python tools/export_model.py \
    -c /content/rec_finetune.yml \
    -o Global.pretrained_model=/content/output/rec/best_accuracy \
       Global.save_inference_dir=/content/output/rec/inference/

print("推論模型已匯出！")
!ls -la /content/output/rec/inference/

In [ ]:
# 轉 ONNX
!mkdir -p /content/output/rec/onnx

!python -m paddle2onnx.convert \
    --model_dir /content/output/rec/inference/ \
    --model_filename inference.pdmodel \
    --params_filename inference.pdiparams \
    --save_file /content/output/rec/onnx/ch_PP-OCRv4_rec_infer.onnx \
    --opset_version 11 \
    --enable_onnx_checker True

print("ONNX 轉換完成！")
!ls -la /content/output/rec/onnx/

In [ ]:
# 驗證 ONNX 模型
import onnxruntime as ort
session = ort.InferenceSession("/content/output/rec/onnx/ch_PP-OCRv4_rec_infer.onnx")
for inp in session.get_inputs():
    print(f"Input:  {inp.name} shape={inp.shape} type={inp.type}")
for out in session.get_outputs():
    print(f"Output: {out.name} shape={out.shape} type={out.type}")
print("\nONNX 模型驗證通過！")

## 9. 下載模型

In [ ]:
# 儲存到 Google Drive
!cp -r /content/output/rec/ /content/drive/MyDrive/ocr_project/output_rec/
print("模型已儲存到 Google Drive！")

In [ ]:
# 直接下載 ONNX
from google.colab import files
files.download("/content/output/rec/onnx/ch_PP-OCRv4_rec_infer.onnx")